In [ ]:
import pathlib
import warnings

import numpy as np
import pandas as pd
import shapely

warnings.filterwarnings("ignore")

# Pre-processing

Before starting the actual analysis, it is important to verify and prepare your data.
This notebook covers:

- **Validating** that trajectories lie within the walkable area
- **Filtering** data by geometry, time, or pedestrian ID

First, we set up the data needed for the examples (see [Analysis Setup](measurement_setup) for details):


In [ ]:
from pedpy import (
    MeasurementArea,
    MeasurementLine,
    TrajectoryData,
    TrajectoryUnit,
    WalkableArea,
    load_trajectory,
)

walkable_area = WalkableArea(
    [
        (3.5, -2),
        (3.5, 8),
        (-3.5, 8),
        (-3.5, -2),
    ],
    obstacles=[
        [
            (-0.7, -1.1),
            (-0.25, -1.1),
            (-0.25, -0.15),
            (-0.4, 0.0),
            (-2.8, 0.0),
            (-2.8, 6.7),
            (-3.05, 6.7),
            (-3.05, -0.3),
            (-0.7, -0.3),
            (-0.7, -1.0),
        ],
        [
            (0.25, -1.1),
            (0.7, -1.1),
            (0.7, -0.3),
            (3.05, -0.3),
            (3.05, 6.7),
            (2.8, 6.7),
            (2.8, 0.0),
            (0.4, 0.0),
            (0.25, -0.15),
            (0.25, -1.1),
        ],
    ],
)

measurement_area = MeasurementArea([(-0.4, 0.5), (0.4, 0.5), (0.4, 1.3), (-0.4, 1.3)])
measurement_line = MeasurementLine([(0.4, 0), (-0.4, 0)])

traj = load_trajectory(
    trajectory_file=pathlib.Path("demo-data/bottleneck/040_c_56_h-.txt"),
    default_unit=TrajectoryUnit.METER,
)

## Trajectory validation

An important step before starting the analysis is to verify that all trajectories lie within the constructed {class}`walkable area <geometry.WalkableArea>`.
Otherwise, you might get errors.
*PedPy* provides a function to test your trajectories, and offers also a function to get all invalid trajectories:


In [ ]:
from pedpy import get_invalid_trajectory, is_trajectory_valid

print(f"Trajectory is valid: {is_trajectory_valid(traj_data=traj, walkable_area=walkable_area)}")
get_invalid_trajectory(traj_data=traj, walkable_area=walkable_area)

**For demonstration purposes, wrongly place the obstacle s.th. some pedestrian walk through it!**

We now create a faulty geometry, s.th. you can see how the result would like.
Therefore, the right obstacle will be moved a bit towards the center of the bottlneck:


In [ ]:
from pedpy import WalkableArea

walkable_area_faulty = WalkableArea(
    # complete area
    [
        (3.5, -2),
        (3.5, 8),
        (-3.5, 8),
        (-3.5, -2),
    ],
    obstacles=[
        # left barrier
        [
            (-0.7, -1.1),
            (-0.25, -1.1),
            (-0.25, -0.15),
            (-0.4, 0.0),
            (-2.8, 0.0),
            (-2.8, 6.7),
            (-3.05, 6.7),
            (-3.05, -0.3),
            (-0.7, -0.3),
            (-0.7, -1.0),
        ],
        # right barrier is too close to the middle
        [
            (0.15, -1.1),
            (0.6, -1.1),
            (0.6, -0.3),
            (3.05, -0.3),
            (3.05, 6.7),
            (2.8, 6.7),
            (2.8, 0.0),
            (0.3, 0.0),
            (0.15, -0.15),
            (0.15, -1.1),
        ],
    ],
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_measurement_setup

ax = plot_measurement_setup(
    traj=traj,
    walkable_area=walkable_area_faulty,
    traj_alpha=0.5,
    traj_width=1,
    hole_color="lightgrey",
)
ax.set_xlim([-1, 1])
ax.set_ylim([-1, 1])
ax.set_xticks([-1, -0.5, 0, 0.5, 1])
ax.set_yticks([-1, -0.5, 0, 0.5, 1])
plt.show()

If you get any invalid trajectories, you should check whether you constructed your {class}`walkable area <geometry.WalkableArea>` correctly.
In some cases you will get such errors when you have head trajectories, and the pedestrian lean over the obstacles.
Then you need to prepare your data before you can start your analysis.


In [ ]:
from pedpy import get_invalid_trajectory, is_trajectory_valid

print(f"Trajectory is valid: {is_trajectory_valid(traj_data=traj, walkable_area=walkable_area_faulty)}")
get_invalid_trajectory(traj_data=traj, walkable_area=walkable_area_faulty)

## Filtering the data

Until now, we used complete trajectories, but sometimes not all the data is relevant for the analysis. 
If the data comes from larger simulation or experiments you may be only interested in data close to your region of interest or data in a specific time range.

As *PedPy* builds up on *Pandas* as data container, the filtering methods from *Pandas* can also be used here.
More information on filtering and merging with *Pandas* can be found here: [filtering](https://pandas.pydata.org/pandas-docs/version/2.0/user_guide/indexing.html) & [merging](https://pandas.pydata.org/pandas-docs/version/2.0/user_guide/merging.html).


### Geometric filtering

First, we want to filter the data by geometrical principles, therefor we combine the capabilities of *Pandas* and *Shapely*.


#### Data inside Polygon

In the first case, we are only interested in {class}`trajectory data<trajectory_data.TrajectoryData>` inside the known {class}`measurement area <geometry.MeasurementArea>`.


In [ ]:
import shapely

bottleneck = shapely.Polygon([(0.25, 0), (0.25, -1), (-0.25, -1), (-0.25, 0)])
leaving_area = shapely.Polygon([(-3, -1), (-3, -2), (3, -2), (3, -1)])

In [ ]:
data_inside_ma = traj.data[shapely.within(traj.data.point, bottleneck)]

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_measurement_setup

ax = plot_measurement_setup(
    traj=TrajectoryData(data_inside_ma, frame_rate=traj.frame_rate),
    walkable_area=walkable_area,
    measurement_areas=[MeasurementArea(bottleneck)],
    traj_alpha=0.7,
    traj_width=0.4,
    ml_width=1,
    ma_alpha=0.1,
    ma_line_width=1,
)
ax.set_xlim([-0.75, 0.75])
ax.set_ylim([-1.5, 0.5])
ax.set_aspect("equal")
plt.show()

#### Data outside Polygon

Secondly, we want to filter the data, such that the result contains only data which is outside a given area. 
In our case we want to remove all data behind the bottleneck, here called leaving area:


In [ ]:
data_outside_leaving_area = traj.data[~shapely.within(traj.data.point, leaving_area)]

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_measurement_setup

ax = plot_measurement_setup(
    traj=TrajectoryData(data_outside_leaving_area, frame_rate=traj.frame_rate),
    walkable_area=walkable_area,
    measurement_areas=[MeasurementArea(leaving_area)],
    traj_alpha=0.7,
    traj_width=0.4,
    ml_width=1,
    ma_alpha=0.1,
    ma_line_width=1,
).set_aspect("equal")

plt.show()

#### Data close to line

It is not only possible to check whether a point is within a given polygon, it is also possible to check if the distance to a given geometrical object is below a given threshold.
Here we want all the data that is within 1m of the {class}`measurement line <geometry.MeasurementLine>` at the entrance of the bottleneck:


In [ ]:
data_close_ma = traj.data[shapely.dwithin(traj.data.point, measurement_line.line, 1)]

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_measurement_setup

ax = plot_measurement_setup(
    traj=TrajectoryData(data_close_ma, frame_rate=traj.frame_rate),
    walkable_area=walkable_area,
    measurement_lines=[measurement_line],
    traj_alpha=0.7,
    traj_width=0.4,
    ml_width=1,
    ma_alpha=0.1,
    ma_line_width=1,
)
ax.set_xlim([-1.5, 1.5])
ax.set_ylim([-2.5, 1.5])
ax.set_aspect("equal")
plt.show()

### Time based filtering

It is not only possible to filter the data by geometrical means, but also depending on time information.
In experiments, you might only be interested in steady state data, this can be also be achieved by slicing the data:


In [ ]:
traj_data_frame_range = traj[300:600]

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_measurement_setup

plot_measurement_setup(
    traj=traj_data_frame_range,
    walkable_area=walkable_area,
    traj_alpha=0.7,
    traj_width=0.4,
).set_aspect("equal")

plt.show()

Alternatively, you could also utilise the Pandas filtering methods:


In [ ]:
data_frame_range = traj.data[traj.data.frame.between(300, 600, inclusive="both")]

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_measurement_setup

plot_measurement_setup(
    traj=TrajectoryData(data_frame_range, frame_rate=traj.frame_rate),
    walkable_area=walkable_area,
    traj_alpha=0.7,
    traj_width=0.4,
).set_aspect("equal")

plt.show()

### ID based filtering

It is also possible to filter the data in a way that only specific pedestrians are contained:


In [ ]:
data_id = traj.data[traj.data.id == 20]

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_measurement_setup

plot_measurement_setup(
    traj=TrajectoryData(data_id, frame_rate=traj.frame_rate),
    walkable_area=walkable_area,
    traj_alpha=0.7,
    traj_width=0.4,
).set_aspect("equal")
plt.show()

It is also possible to filter for multiple ids:


In [ ]:
ids = [10, 20, 30, 40]
data_id = traj.data[traj.data.id.isin(ids)]

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_measurement_setup

plot_measurement_setup(
    traj=TrajectoryData(data_id, frame_rate=traj.frame_rate),
    walkable_area=walkable_area,
    traj_alpha=0.7,
    traj_width=0.4,
).set_aspect("equal")

plt.show()